In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point


In [ ]:
def parse_str_list(x):
    # Check if x is a string like "[0.64863759]"
    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        # Remove brackets and convert to float
        try:
            return float(x.strip('[]'))
        except ValueError:
            return pd.NA
    else:
        # Try direct float conversion
        try:
            return float(x)
        except (ValueError, TypeError):
            return pd.NA

## Create Updated Calibration Result List (2026-02-26)

In [ ]:
# Define the Output Location for the extracted results
OUTPUT_DIR = Path.cwd() 
# We will extract the updated GloFAS5 results in one dataframe. 
# Define the path to the directory containing the Calibration Results
DIR_NEW = Path.cwd() / "GloFASv5_Stations_Overview_260223"
# List the final CSV
csv_file = list(DIR_NEW.glob("*GloFAS*.csv"))[0]  # pick the first file

# Extract the region name from each file path and read the CSV files into a list of DataFrames
glofas5_calib_updated = pd.read_csv(csv_file)

In [22]:
# Load Calib Results
csv_file2 = "GloFASv5_Calibration_Results.csv"
glofas5_calib_full = pd.read_csv(csv_file2) 

In [32]:
# Merge them
# Get unique IDs from each DataFrame
ids_calib = set(glofas5_calib_full['ID'].unique())
ids_stations = set(glofas5_calib_updated['efas'].unique())

# Find IDs only in calibration data
only_in_calib = ids_calib - ids_stations
print(f"IDs only in glofas5_calib_full: {len(only_in_calib)}")
print(f"Example: {list(only_in_calib)[:10]}")

# Find IDs only in stations data
only_in_stations = ids_stations - ids_calib
print(f"\nIDs only in glofas5_stations_attributes: {len(only_in_stations)}")
print(f"Example: {list(only_in_stations)[:10]}")

# Find common IDs
common_ids = ids_calib & ids_stations
print(f"\nCommon IDs: {len(common_ids)}")

# Merge the two DataFrames on the ID column
glofas5_merged = pd.merge(
    glofas5_calib_updated,
    glofas5_calib_full,
    left_on='efas',   # column in glofas5_calib_updated
    right_on='ID',    # column in glofas5_calib_full
    how='outer'
)
print(f"\nMerged DataFrame shape: {glofas5_merged.shape}")


IDs only in glofas5_calib_full: 6
Example: [np.int64(7201), np.int64(7245), np.int64(7188), np.int64(7189), np.int64(7198), np.int64(7199)]

IDs only in glofas5_stations_attributes: 15460
Example: [np.int64(1), np.int64(2), np.int64(3), np.int64(7), np.int64(8), np.int64(10), np.int64(13), np.int64(15), np.int64(16), np.int64(20)]

Common IDs: 5176

Merged DataFrame shape: (20643, 57)


In [52]:
# Save as CSV
output_csv_path = OUTPUT_DIR / "GloFASv5_Calibration_Results_updated.csv"
glofas5_merged.to_csv(output_csv_path, index=False)

# Save as Geojson
geometry = [Point(xy) for xy in zip(glofas5_merged["long"], glofas5_merged["lat_x"])]
gdf = gpd.GeoDataFrame(glofas5_merged, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)
output_geojson_path = OUTPUT_DIR / "GloFASv5_Calibration_Results_updated.geojson"
gdf.to_file(output_geojson_path, driver="GeoJSON")
output_geojson_path = DIR_NEW / "GloFASv5_Calibration_Results_updated.geojson"
gdf.to_file(output_geojson_path, driver="GeoJSON")




---
# Old Overview

# Load Station Calibration Results

In [ ]:
# Define the path to the directory containing the Calibration Results
# We have two different folders from Stefania:
# 1. ./v5_allresultsfromLeonardo_23dec2025/:
#           The folder with the GloFASv5 station files (observation and station metadata per station/basin) + simulated discharge time series
# 2. ./GloFASv5_CALIB_RESULTS_runYYY/:
#           The folder with the GloFASv5 objective function values (NSE, KGE, etc.) for all stations/basins inside a region + calibration plots 

# Define the Output Location for the extracted results
OUTPUT_DIR = Path.cwd() 
# We will extract all regional <Region>_evaluation.csv files from the second folder and all regional results in one dataframe. 
# Define the path to the directory containing the Calibration Results
DIR2 = Path.cwd() / "GloFASv5_CALIB_RESULTS_runYYY"
# List all regional evaluation CSV files in all subfolders of the directory
csv_files = list(DIR2.glob("**/*_evaluation.csv"))
csv_files = [file for file in csv_files if not file.parts[-1].startswith("OLD_")]
csv_files = [file for file in csv_files if "NODATA" not in file.parts[-1]]
csv_files = [file for file in csv_files if "MagdalenaIDEAM" not in file.parts[-2]]

# Extract the region name from each file path and read the CSV files into a list of DataFrames
dfs = []
for csv_file in csv_files:
    region_name = csv_file.stem.split("_evaluation")[0]  # Extract region name from file name
    df = pd.read_csv(csv_file)  # Read the CSV file into a DataFrame
    df["Region"] = region_name  # Add a new column for the region name
    dfs.append(df)  # Append the DataFrame to the list
# Concatenate all DataFrames into a single DataFrame
glofas5_calib_full = pd.concat(dfs, ignore_index=True)
# Extract Unique Regions
glofas5_regions = glofas5_calib_full["Region"].unique()
# Make the list entries in the "modKGE_v5" column numeric, if possible, otherwise set to NA
glofas5_calib_full["modKGE_v5"] = glofas5_calib_full["modKGE_v5"].apply(parse_str_list)
glofas5_calib_full["modKGE_v5"] = pd.to_numeric(glofas5_calib_full["modKGE_v5"], errors='coerce')

# Remove duplicates based on lat lon id, JSD and modKGE
# Define columns to check for duplicates
cols_to_check = ['ID', 'lat', 'lon', 'modKGE_v5',"JSD"]
for col in ['lat', 'lon', 'modKGE_v5',"JSD"]:
    glofas5_calib_full[col] = glofas5_calib_full[col].round(6)

# Drop duplicates based on these columns
glofas5_calib_full = glofas5_calib_full.drop_duplicates(subset=cols_to_check, keep='first')



In [ ]:
# Save the full DataFrame to a CSV file & a Geojson file
output_csv_path = OUTPUT_DIR / "GloFASv5_Calibration_Results.csv"
glofas5_calib_full.to_csv(output_csv_path, index=False)

geometry = [Point(xy) for xy in zip(glofas5_calib_full["lon"], glofas5_calib_full["lat"])]
gdf = gpd.GeoDataFrame(glofas5_calib_full, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)
output_geojson_path = OUTPUT_DIR / "GloFASv5_Calibration_Results.geojson"
gdf.to_file(output_geojson_path, driver="GeoJSON")

## Check Cases Where *_evaluation.csv might be called differently

In [ ]:
# List all folders that might contain evaluation files
all_folders = [f for f in DIR2.glob("**/") if f.is_dir()]

missing_standard_files = {}
for folder in all_folders:
    # Check if the standard *_evaluation.csv exists
    standard_files = list(folder.glob("*_evaluation.csv"))
    
    if not standard_files:  # Standard file is missing
        # List all alternative evaluation files
        alternative_files = list(folder.glob("*_evaluation_*.csv"))
        if alternative_files:
            # Save folder and alternative file names
            missing_standard_files[folder] = [f.name for f in alternative_files]

# Show results
for folder, files in missing_standard_files.items():
    print(f"Folder: {folder}")
    print(f"  Alternative files: {files}\n")

Folder: /mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Calibration/01_Results/from_Stefania/GloFASv5_CALIB_RESULTS_runYYY/Africa_Chad_GuineaGulf_runYYY_COMPLETED
  Alternative files: ['Africa_Chad_GuineaGulf_evaluation_2000-2024.csv', 'Africa_Chad_GuineaGulf_evaluation_ALLyears.csv']

Folder: /mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Calibration/01_Results/from_Stefania/GloFASv5_CALIB_RESULTS_runYYY/Asia_SiberiaWestCoastSiberiaNorthCoastYeniseiLena_runYYY_COMPLETED
  Alternative files: ['Asia_SiberiaWestCoastSiberiaNorthCoastYeniseiLena_evaluation_CHECK.csv']



# Load Station Summary Statistics

In [ ]:
# We will extract all Station Summary Statistics from the first folder
# Define the path to the directory containing the Calibration Results
DIR1 = Path.cwd() / "v5_allresultsfromLeonardo_23dec2025/"
# List all regional evaluation CSV files in all subfolders of the directory
csv_files = list(DIR1.glob("**/station_data.csv"))
csv_files = [file for file in csv_files if "MagdalenaIDEAM" not in file.parts[-4]]
csv_files

[PosixPath('/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Calibration/01_Results/from_Stefania/v5_allresultsfromLeonardo_23dec2025/Africa/Angola_Namibia_SouthInterior/14580/station/station_data.csv'),
 PosixPath('/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Calibration/01_Results/from_Stefania/v5_allresultsfromLeonardo_23dec2025/Africa/Angola_Namibia_SouthInterior/14587/station/station_data.csv'),
 PosixPath('/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Calibration/01_Results/from_Stefania/v5_allresultsfromLeonardo_23dec2025/Africa/Angola_Namibia_SouthInterior/17077/station/station_data.csv'),
 PosixPath('/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Calibration/01_Results/from_Stefania/v5_allresultsfromLeonardo_23dec2025/Africa/Angola_Namibia_SouthInterior/17078/station/station_data.csv'),
 PosixPath('/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/Cal

In [ ]:
# Extract the region name from each file path and read the CSV files into a list of DataFrames
dfs = []
for csv_file in csv_files:
    # Extract gauge_id from folder structure
    gauge_id = csv_file.parts[-3]
    if any(word.lower() in gauge_id.lower() for word in ["removed", "wrong"]):
        continue
    elif "_" in gauge_id:
        gauge_id = gauge_id.split("_")[1]
        gauge_id = int(gauge_id)
    else:
         gauge_id = int(gauge_id)   
    # Read file
    df = pd.read_csv(csv_file)
    # Rename columns to something usable
    df.columns = ["ID", "value"]
    # Set attribute column as index and transpose
    df_wide = df.set_index("ID").T
    # Add gauge_id as index
    df_wide.index = [gauge_id]

    dfs.append(df_wide)

# Combine all gauges into one DataFrame
glofas5_stations_attributes = pd.concat(dfs)

# Convert all columns to numeric where possible
for col in glofas5_stations_attributes.columns:
    glofas5_stations_attributes[col] = pd.to_numeric(glofas5_stations_attributes[col], errors='ignore')

glofas5_stations_attributes = glofas5_stations_attributes.reset_index()
glofas5_stations_attributes = glofas5_stations_attributes.rename(columns={'index': 'ID'})

# Remove duplicates based on lat lon id, JSD and modKGE
# Define columns to check for duplicates
cols_to_check = ['ID', 'StationLat', 'StationLon', 'precip_budyko',"PET_budyko","min_TAvgS","min_AridIdx", "Country code"]
for col in ['StationLat', 'StationLon', 'precip_budyko',"PET_budyko","min_TAvgS","min_AridIdx"]:
    glofas5_stations_attributes[col] = glofas5_stations_attributes[col].round(6)

# Drop duplicates based on these columns
glofas5_stations_attributes = glofas5_stations_attributes.drop_duplicates(subset=cols_to_check, keep='first')

glofas5_stations_attributes

/tmp/ipykernel_1719021/395886121.py:29: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  glofas5_stations_attributes[col] = pd.to_numeric(glofas5_stations_attributes[col], errors='ignore')


ID,ID,StationName,Provider ID,Country code,StationLat,StationLon,Height,Height Units,DrainingArea.km2.Provider,Catchment Area Units,...,Spinup_days,Min_calib_days,Obs_start,Obs_end,Split_date,N_data,precip_budyko,PET_budyko,min_TAvgS,min_AridIdx
0,14580,TATI WEIR (67535411),1077,BW,-21.0000,26.7500,NaN,NaN,NaN,646,...,1095,1460,01/01/1980 00:00,01/10/1993 00:00,01/01/1980 00:00,4646,7363.700468,27387.775307,8.639569,0.264614
1,14587,MOTSETSE (67921012),1077,BW,-20.6500,26.6500,NaN,NaN,NaN,1048,...,1095,1460,01/01/1980 00:00,01/10/1993 00:00,01/01/1980 00:00,4927,7766.792030,27464.260579,8.461508,0.272533
2,17077,OMBUKU (64740002),1077,NaN,-17.2700,13.3100,NaN,NaN,NaN,1620,...,1095,1460,03/02/1980 00:00,29/06/2020 00:00,03/02/1980 00:00,12640,21592.799370,71948.827261,6.143007,0.234293
3,17078,OUSEMA (64932301),1077,NaN,-21.2200,17.1000,NaN,NaN,NaN,4970,...,1095,1460,01/01/1980 00:00,11/02/2004 00:00,24/05/1982 00:00,8179,10067.808025,42188.554144,4.575962,0.205827
4,17079,ROOIBERG (64735002),1077,NaN,-20.4800,14.5900,NaN,NaN,NaN,1570,...,1095,1460,01/01/1980 00:00,05/01/2016 00:00,01/01/1980 00:00,12351,11987.176758,68913.919997,7.512076,0.124639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5397,13993,CONCEIÇÃO DO ARAGUAIA,1104,BR,-8.2694,-49.2594,NaN,NaN,332000.0,km2,...,1095,1460,01/01/1980 00:00,01/04/2023 00:00,23/02/2003 00:00,15759,38870.520603,34313.485057,18.593678,1.033896
5398,14001,XAMBIOÁ,1104,BR,-6.4097,-48.5422,NaN,NaN,377000.0,km2,...,1095,1460,01/01/1980 00:00,01/01/2023 00:00,17/10/2002 00:00,15630,38246.487714,33873.338168,19.819414,1.040576
5399,14003,ARAGUATINS,1104,BR,-5.6514,-48.1325,NaN,NaN,388000.0,km2,...,1095,1460,01/01/1980 00:00,01/01/2023 00:00,01/12/2002 00:00,15675,37717.068646,35467.141194,20.884012,0.942398
5400,6408,Itupiranga,1077,BR,-5.1281,-49.3242,NaN,NaN,731270.0,km2,...,1095,1460,01/01/1980 00:00,01/03/2015 00:00,07/03/1993 00:00,11956,38123.148134,35388.852195,21.274305,0.788251


# Merge the 2 Datasets

In [ ]:
# Get unique IDs from each DataFrame
ids_calib = set(glofas5_calib_full['ID'].unique())
ids_stations = set(glofas5_stations_attributes['ID'].unique())

# Find IDs only in calibration data
only_in_calib = ids_calib - ids_stations
print(f"IDs only in glofas5_calib_full: {len(only_in_calib)}")
print(f"Example: {list(only_in_calib)[:10]}")

# Find IDs only in stations data
only_in_stations = ids_stations - ids_calib
print(f"\nIDs only in glofas5_stations_attributes: {len(only_in_stations)}")
print(f"Example: {list(only_in_stations)[:10]}")

# Find common IDs
common_ids = ids_calib & ids_stations
print(f"\nCommon IDs: {len(common_ids)}")

# Merge the two DataFrames on the ID column
glofas5_merged = pd.merge(
    glofas5_stations_attributes,
    glofas5_calib_full,
    on='ID',
    how='outer'
)
print(f"\nMerged DataFrame shape: {glofas5_merged.shape}")


IDs only in glofas5_calib_full: 10
Example: [np.int64(6148), np.int64(11881), np.int64(331), np.int64(16139), np.int64(15855), np.int64(13617), np.int64(598), np.int64(2838), np.int64(7165), np.int64(6718)]

IDs only in glofas5_stations_attributes: 115
Example: [np.int64(17935), np.int64(15379), np.int64(15380), np.int64(15382), np.int64(15383), np.int64(15384), np.int64(15385), np.int64(15386), np.int64(15387), np.int64(5660)]

Common IDs: 5172

Merged DataFrame shape: (5381, 100)


In [ ]:
# Get duplicates of glofas5_calib_full
duplicates = glofas5_merged[glofas5_merged.duplicated(subset=['ID'], keep=False)]['ID'].unique()
duplicates

array([  133,  1966,  4471,  6607,  6620,  6622,  6650,  6657,  6660,
        6669,  6673,  6794,  6797,  6800,  6802,  6804,  6805,  6807,
        6810,  6812,  6815,  6821,  6823,  6829,  6841,  6845,  6846,
        6853,  6865,  6866,  6928,  7330,  7777,  7778,  7780,  7801,
        7809,  7811,  7812,  7819,  7826,  8593,  8595,  8596,  8597,
        8603,  8605,  8615,  8616,  8621,  8633,  8635,  8641,  8645,
        8646,  8648,  8655,  8657,  8659,  8661,  8672,  8675,  8680,
        8681,  8683,  8686,  8694,  8695,  8699,  8703,  8704,  8706,
        8707,  8708,  8709,  8797,  9087, 15140, 17077, 17079, 17089,
       17091, 17097, 17098])

In [ ]:
# Save the extended DataFrame with catchment attributes to a CSV file & a Geojson file
output_csv_path = OUTPUT_DIR / "GloFASv5_Calibration_Results_with_attr.csv"
glofas5_merged.to_csv(output_csv_path, index=False)

geometry = [Point(xy) for xy in zip(glofas5_merged["lon"], glofas5_merged["lat"])]
gdf = gpd.GeoDataFrame(glofas5_merged, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)
output_geojson_path = OUTPUT_DIR / "GloFASv5_Calibration_Results_with_attr.geojson"
gdf.to_file(output_geojson_path, driver="GeoJSON")